In [1]:
import joblib
import pandas as pd

# Crop model
crop_model = joblib.load("../models/crop_model.pkl")
le_crop = joblib.load("../models/crop_label_encoder.pkl")

# Disease model
disease_model = joblib.load("../models/disease_model.pkl")
le_disease = joblib.load("../models/disease_label_encoder.pkl")

# Irrigation model
irrig_model = joblib.load("../models/irrigation_model.pkl")
le_irrig_crop = joblib.load("../models/irrig_crop_encoder.pkl")
le_soil = joblib.load("../models/soil_encoder.pkl")
le_irrig = joblib.load("../models/irrig_label_encoder.pkl")

# Yield model
yield_model = joblib.load("../models/yield_model.pkl")
le_yield_crop = joblib.load("../models/yield_crop_encoder.pkl")
le_yield = joblib.load("../models/yield_label_encoder.pkl")

print("All models loaded")


c:\Users\91974\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


All models loaded


c:\Users\91974\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\91974\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
def recommend_crop(N, P, K, temp, humidity, ph, rainfall):

    data = pd.DataFrame([{
        "N": N,
        "P": P,
        "K": K,
        "temperature": temp,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall
    }])

    probs = crop_model.predict_proba(data)[0]
    classes = le_crop.classes_

    ranked = sorted(
        zip(classes, probs),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked


In [3]:
def recommend_crop(N, P, K, temp, humidity, ph, rainfall):

    data = pd.DataFrame([{
        "N": N,
        "P": P,
        "K": K,
        "temperature": temp,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall
    }])

    probs = crop_model.predict_proba(data)[0]
    classes = le_crop.classes_

    ranked = sorted(
        zip(classes, probs),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked


In [4]:
def disease_risk(temp, humidity, rainfall, ph):

    data = pd.DataFrame([{
        "temperature": temp,
        "humidity": humidity,
        "rainfall": rainfall,
        "soil_pH": ph
    }])

    pred = disease_model.predict(data)[0]
    return le_disease.inverse_transform([pred])[0]


In [5]:
def irrigation_advice(crop, temp, humidity, rainfall, soil_condition):

    crop_enc = le_irrig_crop.transform([crop])[0]
    soil_enc = le_soil.transform([soil_condition])[0]

    data = pd.DataFrame([{
        "crop": crop_enc,
        "temperature": temp,
        "humidity": humidity,
        "rainfall": rainfall,
        "soil_condition": soil_enc
    }])

    pred = irrig_model.predict(data)[0]
    return le_irrig.inverse_transform([pred])[0]


In [6]:
def yield_potential(crop, inputs):

    crop_enc = le_yield_crop.transform([crop])[0]

    data = pd.DataFrame([{
        "crop": crop_enc,
        "rainfall": inputs["rainfall"],
        "temperature": inputs["temperature"],
        "humidity": inputs["humidity"],
        "ph": inputs["ph"],
        "N": inputs["N"],
        "P": inputs["P"],
        "K": inputs["K"]
    }])

    # enforce training column order
    data = data[["crop","rainfall","temperature","humidity","ph","N","P","K"]]

    pred = yield_model.predict(data)[0]
    return le_yield.inverse_transform([pred])[0]


In [7]:
def agrobrain_advisory(inputs):

    # Stage 1 — Ranked crop recommendation
    ranked_crops = recommend_crop(
        inputs["N"], inputs["P"], inputs["K"],
        inputs["temperature"], inputs["humidity"],
        inputs["ph"], inputs["rainfall"]
    )

    # Extract top 2 crops
    top_crop = ranked_crops[0][0]
    second_crop = ranked_crops[1][0]

    # Stage 2 — Management advice for TOP crop only
    disease = disease_risk(
        inputs["temperature"],
        inputs["humidity"],
        inputs["rainfall"],
        inputs["ph"]
    )

    irrigation = irrigation_advice(
        top_crop,
        inputs["temperature"],
        inputs["humidity"],
        inputs["rainfall"],
        inputs["soil_condition"]
    )

    return {
        "ranked_crops": ranked_crops,
        "top_crop": top_crop,
        "second_crop": second_crop,
        "disease_risk": disease,
        "irrigation": irrigation
    }

In [8]:
sample_input = {
    "N": 80,
    "P": 40,
    "K": 40,
    "temperature": 30,
    "humidity": 70,
    "ph": 6.5,
    "rainfall": 120,
    "soil_condition": "Moist"
}

result = agrobrain_advisory(sample_input)
print(result)


{'ranked_crops': [('coffee', np.float64(0.615)), ('jute', np.float64(0.19)), ('banana', np.float64(0.06)), ('mango', np.float64(0.045)), ('papaya', np.float64(0.04)), ('maize', np.float64(0.025)), ('coconut', np.float64(0.01)), ('pigeonpeas', np.float64(0.01)), ('muskmelon', np.float64(0.005)), ('apple', np.float64(0.0)), ('blackgram', np.float64(0.0)), ('chickpea', np.float64(0.0)), ('cotton', np.float64(0.0)), ('grapes', np.float64(0.0)), ('kidneybeans', np.float64(0.0)), ('lentil', np.float64(0.0)), ('mothbeans', np.float64(0.0)), ('mungbean', np.float64(0.0)), ('orange', np.float64(0.0)), ('pomegranate', np.float64(0.0)), ('rice', np.float64(0.0)), ('watermelon', np.float64(0.0))], 'top_crop': 'coffee', 'second_crop': 'jute', 'disease_risk': 'High', 'irrigation': '3-4 days'}


In [9]:
sample_input = {
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 28,
    "humidity": 85,
    "ph": 6.4,
    "rainfall": 220,
    "soil_condition": "Wet"
}

print(agrobrain_advisory(sample_input))


{'ranked_crops': [('rice', np.float64(0.56)), ('banana', np.float64(0.125)), ('jute', np.float64(0.115)), ('papaya', np.float64(0.095)), ('coffee', np.float64(0.04)), ('coconut', np.float64(0.02)), ('mungbean', np.float64(0.02)), ('muskmelon', np.float64(0.02)), ('watermelon', np.float64(0.005)), ('apple', np.float64(0.0)), ('blackgram', np.float64(0.0)), ('chickpea', np.float64(0.0)), ('cotton', np.float64(0.0)), ('grapes', np.float64(0.0)), ('kidneybeans', np.float64(0.0)), ('lentil', np.float64(0.0)), ('maize', np.float64(0.0)), ('mango', np.float64(0.0)), ('mothbeans', np.float64(0.0)), ('orange', np.float64(0.0)), ('pigeonpeas', np.float64(0.0)), ('pomegranate', np.float64(0.0))], 'top_crop': 'rice', 'second_crop': 'banana', 'disease_risk': 'High', 'irrigation': '3-4 days'}
